In [ ]:
"""
Objective: Extract macroeconomic time series from the Federal Reserve API (FRED),
harmonize data frequency into complete quarters, and export the clean dataset.
"""

import pandas as pd
import pandas_datareader.data as web
import datetime
import warnings
import json
import os
import sys

warnings.filterwarnings('ignore')

def load_configuration(file_path: str = "config_en.json") -> dict:
    """Reads parameters and variables from an external JSON file."""
    if not os.path.exists(file_path):
        print(f"[ERROR] Configuration file '{file_path}' not found.")
        sys.exit(1)
        
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            config = json.load(file)
            if "series_fred" not in config or "parametros_gerais" not in config:
                print("[ERROR] Missing required keys in config.json.")
                sys.exit(1)
            return config
    except Exception as e:
        print(f"[ERROR] Failed to read JSON configuration file: {e}")
        sys.exit(1)

def extract_fred_data(series_dict: dict, start_date: str, end_date: str) -> pd.DataFrame:
    """Consumes the FRED API to download time series and consolidates them into a single DataFrame."""
    print(f"[*] Starting extraction of {len(series_dict)} series from FRED...")
    series_list = []
    
    for ticker, column_name in series_dict.items():
        try:
            print(f" - Downloading ticker: {ticker} ({column_name})...")
            series = web.DataReader(ticker, 'fred', start_date, end_date)
            clean_series = pd.to_numeric(series[ticker], errors='coerce')
            clean_series.name = column_name
            series_list.append(clean_series)
        except Exception as e:
            print(f"[WARNING] Failed to download series {ticker}: {e}")
            
    if not series_list:
        print("[ERROR] No series were successfully downloaded from FRED.")
        sys.exit(1)
            
    return pd.concat(series_list, axis=1)

def harmonize_to_quarterly(df: pd.DataFrame) -> pd.DataFrame:
    """Converts daily/monthly data to quarterly averages and drops the current incomplete quarter."""
    print("[*] Harmonizing dataset to quarterly frequency (Mean)...")
    quarterly_df = df.resample('QS').mean()
    
    current_quarter = pd.Timestamp(datetime.datetime.today()).to_period('Q').start_time
    if current_quarter in quarterly_df.index:
        quarterly_df = quarterly_df.drop(current_quarter)
        print(f"[*] Current quarter ({current_quarter.date()}) removed to prevent partial data bias.")
    
    return quarterly_df.dropna(how='all')

if __name__ == "__main__":
    # 1. Load parameters
    config = load_configuration("config_en.json")
    start = config["parametros_gerais"]["start_date"]
    end = datetime.datetime.today().strftime('%Y-%m-%d')
    output_file = config["parametros_gerais"]["output_file"]
    
    # 2. Run data pipeline
    raw_df = extract_fred_data(config["series_fred"], start, end)
    final_df = harmonize_to_quarterly(raw_df)
    
    # 3. Export results
    final_df.to_csv(output_file, index=True)
    
    print(f"\n[SUCCESS] Final dataset exported as '{output_file}'.")
    print(f"[*] Macro Panel Dimensions: {final_df.shape[0]} quarters and {final_df.shape[1]} variables.")